In [1]:
import pandas as pd
import numpy as np

In [2]:
business_interruption_file_path = "dataset/srcsc-2026-claims-business-interruption.xlsx"
claim_cargo_file_path = "dataset/srcsc-2026-claims-cargo.xlsx"
claims_equipment_failure_file_path = "dataset/srcsc-2026-claims-equipment-failure.xlsx"
claims_workers_comp_file_path = "dataset/srcsc-2026-claims-workers-comp.xlsx"

In [23]:
def business_interruption_clean_sheet_1(df):
    '''
    Inner functions, needed for tedious, repeatative work
    '''
    def ratio(df, column_name):
        """
        Cleans a column to bring values toward a 0-1 range:
        - 0 to 1: No change
        - 0 to -1: Absolute value
        - > 1 or < -1: Divide by 100
        """
        # Use a local variable for readability
        col = df[column_name]
        
        conditions = [
            (col >= 0) & (col <= 1),   # Pass
            (col < 0) & (col >= -1),  # Small negative -> Absolute
            (col < -1),               # Large negative -> /100
            (col > 1)                 # Large positive -> /100
        ]

        choices = [
            col,          # Keep original
            col.abs(),    # Absolute value
            col / 100,    # Divide by 100
            col / 100     # Divide by 100
        ]

        # Apply logic and return the modified column (or update df)
        df[column_name] = np.select(conditions, choices, default=col)
        
        return df

    def range(df, column_name, start, end):
        # 1. Absolute value to fix negatives
        df[column_name] = df[column_name].abs()
        
        # 2. Round to the nearest integer (e.g., 3.7 -> 4.0)
        # Use .round(0) then convert to Int64 to handle potential NaNs safely
        df[column_name] = df[column_name].round(0)
        
        # 3. Constrain to the 0-4 range
        df[column_name] = df[column_name].clip(lower = start, upper = end)
        
        # 4. Convert to integer type
        # 'Int64' (capital I) allows for NaN values, unlike 'int64'
        df[column_name] = df[column_name].astype('Int64')
        
        return df

    '''
    The below few starts with column cleaning the oddities.
    
    Refer to Data dictionary for why this cleaning is done the way it is
    '''
    # 'policy_id' cleaning
    df['policy_id'] = df['policy_id'].str[:9]
    
    # 'solar_system' cleaning
    valid_names = [
        'Helionis Cluster',
        'Epsilon',
        'Zeta'
    ]
    pattern = f"({'|'.join(valid_names)})"
    df['solar_system'] = df['solar_system'].str.extract(pattern, expand=False)
    
    # Ensure correct ratio for 'production_load'
    df = ratio(df, 'production_load')
    
    # Ensure correct range for 'energy_backup_score'
    df = range(df, 'energy_backup_score', 1, 5)
    
    # Ensure correct ratio for 'supply_chain_index'
    df = ratio(df, 'supply_chain_index')
    
    # Ensure correct range for 'avg_crew_exp'
    df = range(df, 'avg_crew_exp', 1, 30)
    
    # Ensure correct range for 'maintenance_freq'
    df = range(df, 'maintenance_freq', 0, 6)
    
    # Ensure correct range for 'safety_compliance'
    df = range(df, 'safety_compliance', 1, 5)
    
    # Ensure correct ratio for 'exposure'
    df = ratio(df, 'exposure')
    
    # Ensure correct range for 'claim_count'
    df = range(df, 'claim_count', 0, 4)

    return df

In [24]:
df = pd.read_excel(business_interruption_file_path)
df = business_interruption_clean_sheet_1(df)

In [ ]:
df